# 🔍 Notebook 01: Leitura Diagnóstica, Ingestão em Chunks e Auditoria de Qualidade
### Projeto: Análise e Diagnóstico de Acidentes de Trânsito em Bauru/SP (2015–2026)

Este notebook realiza a etapa inicial do pipeline de dados:
1. Inspeção diagnóstica dos arquivos estaduais volumosos do **Infosiga SP**;
2. Ingestão em lotes (*chunks*) filtrando pelo município de **Bauru/SP** (Código IBGE: `3506003`);
3. Auditoria de valores nulos, tipagem e consistência temporal;
4. Identificação do marco metodológico do Infosiga (2015-2018 vs. 2019-2026).

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

print("Diretório de trabalho:", os.getcwd())
data_files = sorted(glob.glob('../data/sinistros_*.csv'))
print("Arquivos de sinistros encontrados:", data_files)

## 1. Ingestão em Lotes (Chunks)
Como os arquivos brutos estaduais somam centenas de megabytes, utilizamos o processamento em chunks de 50.000 linhas para filtrar apenas as ocorrências de Bauru.

In [ ]:
dfs = []
for file_path in data_files:
    print(f"Processando {file_path}...")
    for chunk in pd.read_csv(file_path, sep=';', encoding='latin1', low_memory=False, chunksize=50000):
        # Filtro pelo código IBGE ou nome do município
        filtro = (chunk['cod_ibge'].astype(str) == '3506003') | (chunk['municipio'].astype(str).str.upper() == 'BAURU')
        chunk_bauru = chunk[filtro]
        if len(chunk_bauru) > 0:
            dfs.append(chunk_bauru)

df_raw = pd.concat(dfs, ignore_index=True)
print(f"Total de registros de Bauru extraídos: {len(df_raw)}")
df_raw.head()

## 2. Auditoria de Tipos de Registro e Quebra Metodológica
No Infosiga SP, os registros entre 2015 e 2018 contemplavam prioritariamente sinistros com **vítimas fatais**. A partir de 2019, o sistema passou a registrar também sinistros **não fatais** e notificações sem lesões.

In [ ]:
print("Distribuição por Tipo de Registro:")
print(df_raw['tipo_registro'].value_counts(dropna=False))

print("\nVolume por Ano de Sinistro:")
print(df_raw['ano_sinistro'].value_counts().sort_index())

## 3. Auditoria de Nulos e Coordenadas
Verificação do percentual de preenchimento das coordenadas geográficas e das principais variáveis viárias.

In [ ]:
nulos = df_raw.isnull().sum()
pct_nulos = (nulos / len(df_raw)) * 100
tabela_nulos = pd.DataFrame({'Total Nulos': nulos, 'Percentual (%)': pct_nulos})
tabela_nulos.sort_values(by='Total Nulos', ascending=False).head(15)

## 4. Conclusão Diagnóstica
- O conjunto de Bauru totaliza **18.938 sinistros** brutos.
- A série histórica possui estabilidade nos óbitos contínuos desde 2015 e grande riqueza de variáveis a partir de 2019.
- O próximo passo é o tratamento dos tipos de dados, limpeza das coordenadas e feature engineering no `02-limpeza.ipynb`.